# SW-WAVENET: Learning Representation from Spectrogram and Wavegram Using Wavenet for Anomalous Sound Detection
### https://ieeexplore.ieee.org/document/10096742

## DataLoader

---

In [1]:
import os
import torch
import torchaudio
from torch.utils.data import Dataset


def build_id_mapping(train_dir):
    raw_ids = []

    for fname in os.listdir(train_dir):
        if fname.endswith(".wav"):
            parts = fname.split('_')
            machine_id = int(parts[2])
            raw_ids.append(machine_id)

    unique_ids = sorted(list(set(raw_ids)))
    id_to_index = {id_: i for i, id_ in enumerate(unique_ids)}

    print("Training IDs:", unique_ids)
    print("ID to Index mapping:", id_to_index)
    return id_to_index


class SafeMachineDataset(Dataset):
    def __init__(self, folder_path, id_to_index, expected_samples=160000):
        self.paths = []
        self.labels = []
        self.raw_ids = []
        self.expected_samples = expected_samples

        for fname in sorted(os.listdir(folder_path)):
            if fname.endswith(".wav"):

                parts = fname.split('_')
                machine_id = int(parts[2])

                # Skip unseen IDs
                if machine_id not in id_to_index:
                    continue

                self.paths.append(os.path.join(folder_path, fname))
                self.labels.append(id_to_index[machine_id])
                self.raw_ids.append(machine_id)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            signal, sr = torchaudio.load(path)
        except Exception:
            import soundfile as sf
            data, sr = sf.read(path)
            if len(data.shape) > 1:
                data = data[:, 0]
            signal = torch.from_numpy(data).float().unsqueeze(0)

        if signal.shape[1] < self.expected_samples:
            pad = self.expected_samples - signal.shape[1]
            signal = torch.nn.functional.pad(signal, (0, pad))
        else:
            signal = signal[:, :self.expected_samples]

        return signal, torch.tensor(self.labels[idx], dtype=torch.long)


## Model Classes
---

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import math


# =========================
# ArcFace
# =========================
class ArcFaceLayer(nn.Module):
    def __init__(self, in_features, out_classes, s=30.0, m=0.7):
        super().__init__()
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, features, labels=None):

        cosine = F.linear(F.normalize(features),
                          F.normalize(self.weight))

        if labels is None:
            return cosine * self.s

        sine = torch.sqrt(torch.clamp(1.0 - cosine ** 2, min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th,
                          phi,
                          cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)

        output = (one_hot * phi) + ((1 - one_hot) * cosine)
        return output * self.s


# =========================
# WaveNet Block
# =========================
class DilatedResidualBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()

        self.conv_filter = nn.Conv1d(channels, channels, 2, dilation=dilation)
        self.conv_gate = nn.Conv1d(channels, channels, 2, dilation=dilation)

        self.conv_res = nn.Conv1d(channels, channels, 1)
        self.conv_skip = nn.Conv1d(channels, channels, 1)

        self.bn = nn.BatchNorm1d(channels)

    def forward(self, x):
        pad = self.conv_filter.dilation[0]
        x_padded = F.pad(x, (pad, 0))

        filter_out = torch.tanh(self.conv_filter(x_padded))
        gate_out = torch.sigmoid(self.conv_gate(x_padded))

        out = filter_out * gate_out
        out = self.bn(out)

        res = self.conv_res(out) + x
        skip = self.conv_skip(out)

        return res, skip


class ModifiedWaveNet(nn.Module):
    def __init__(self, in_channels=128, hidden=512):
        super().__init__()

        self.causal_conv = nn.Conv1d(in_channels, hidden, 2)

        self.blocks = nn.ModuleList([
            DilatedResidualBlock(hidden, d)
            for _ in range(3)
            for d in [1, 2, 4, 8]
        ])

        self.bn = nn.BatchNorm1d(hidden)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.final = nn.Conv1d(hidden, 128, 1)

    def forward(self, x):
        x = F.pad(x, (1, 0))
        x = self.causal_conv(x)

        skips = []
        for block in self.blocks:
            x, skip = block(x)
            skips.append(skip)

        out = sum(skips)
        out = self.relu(self.bn(out))
        out = self.pool(out)
        out = self.final(out)

        return out.squeeze(-1)


# =========================
# Full Model
# =========================
class SW_WaveNet(nn.Module):
    def __init__(self, num_classes, sample_rate=16000):
        super().__init__()

        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )

        self.db = torchaudio.transforms.AmplitudeToDB()
        self.wavegram = nn.Conv1d(1, 128, 1024, 512, padding=512)

        self.wavenet_spec = ModifiedWaveNet(128)
        self.wavenet_wave = ModifiedWaveNet(128)

        self.arcface = ArcFaceLayer(256, num_classes)

    def forward(self, raw_audio, labels=None):

        mel = self.db(self.mel(raw_audio.squeeze(1)))
        wave = self.wavegram(raw_audio)

        vec1 = self.wavenet_spec(mel)
        vec2 = self.wavenet_wave(wave)

        combined = torch.cat([vec1, vec2], dim=1)

        return self.arcface(combined, labels)

## Traning Section
---

In [3]:
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

def train_model(train_dir,
                id_mapping,
                device,
                epochs=120,
                batch_size=64,
                lr=1e-4,
                save_dir="checkpoints"):

    os.makedirs(save_dir, exist_ok=True)

    dataset = SafeMachineDataset(train_dir, id_mapping)
    loader = DataLoader(dataset,
                        batch_size=batch_size,
                        shuffle=True,
                        drop_last=True)

    model = SW_WaveNet(num_classes=len(id_mapping)).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.CrossEntropyLoss()

    print("\nTraining Started...\n")

    for epoch in range(epochs):

        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for audio, labels in loader:

            audio, labels = audio.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(audio, labels)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        train_acc = correct / total
        avg_loss = total_loss / len(loader)

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"| Loss: {avg_loss:.4f} "
              f"| Acc: {train_acc:.4f}")

        # ✅ Save every 30 epochs
        if (epoch + 1) % 30 == 0:
            checkpoint_path = os.path.join(
                save_dir,
                f"sw_wavenet_epoch_{epoch+1}.pth"
            )

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "id_mapping": id_mapping
            }, checkpoint_path)

            print(f"✅ Checkpoint saved at epoch {epoch+1}")

    # ✅ Save final model
    final_path = os.path.join(save_dir, "sw_wavenet_final.pth")
    torch.save(model.state_dict(), final_path)

    print("\nTraining Completed.")
    print(f"Final model saved at: {final_path}")

    return model

### Execute Training or Load Pre-Trained Model
---

In [4]:
import os
import torch

# ============================================================
# PATH CONFIGURATION (Update paths as needed)
# ============================================================
BASE_DIR = "/kaggle/input/datasets/vuppalaadithyasairam/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection"
TRAIN_DIR = os.path.join(BASE_DIR, "train-normal")
TEST_NORMAL_DIR = os.path.join(BASE_DIR, "test-normal")
ANOMALY_DIR = os.path.join(BASE_DIR, "anomaly")

# Path to pre-trained model (TorchScript .pt or PyTorch .pth)
MODEL_PATH = "/kaggle/input/sw-wavenet-model/sw_wavenet_traced_cpu.pt"
LOAD_PRETRAINED = True  # Set to False if you want to train from scratch

# ============================================================
# SETUP & LOAD MODEL
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

id_mapping = build_id_mapping(TRAIN_DIR)

if LOAD_PRETRAINED and os.path.exists(MODEL_PATH):
    print(f"Loading pre-trained model from: {MODEL_PATH}")
    model = SW_WaveNet(num_classes=len(id_mapping)).to(device)
    if MODEL_PATH.endswith(".pt"):
        # Load weights from TorchScript and transfer to native PyTorch model (GPU/CPU compatible)
        traced = torch.jit.load(MODEL_PATH, map_location="cpu")
        state_dict = {k.replace("model.", ""): v for k, v in traced.state_dict().items()}
        model.load_state_dict(state_dict)
    else:
        state_dict = torch.load(MODEL_PATH, map_location=device)
        if "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"Model loaded successfully on {device}!")
else:
    print("Training model from scratch...")
    model = train_model(TRAIN_DIR, id_mapping, device, epochs=150)


Training IDs: [0, 2, 4]

Training Started...

Epoch [1/150] | Loss: 4.0240 | Acc: 0.6429
Epoch [2/150] | Loss: 0.2801 | Acc: 0.9629
Epoch [3/150] | Loss: 0.2571 | Acc: 0.9665
Epoch [4/150] | Loss: 0.1043 | Acc: 0.9857
Epoch [5/150] | Loss: 0.1189 | Acc: 0.9862
Epoch [6/150] | Loss: 0.1256 | Acc: 0.9884
Epoch [7/150] | Loss: 0.1275 | Acc: 0.9835
Epoch [8/150] | Loss: 0.0793 | Acc: 0.9862
Epoch [9/150] | Loss: 0.0806 | Acc: 0.9893
Epoch [10/150] | Loss: 0.1012 | Acc: 0.9893
Epoch [11/150] | Loss: 0.0420 | Acc: 0.9933
Epoch [12/150] | Loss: 0.0291 | Acc: 0.9946
Epoch [13/150] | Loss: 0.2542 | Acc: 0.9759
Epoch [14/150] | Loss: 0.1759 | Acc: 0.9835
Epoch [15/150] | Loss: 0.1003 | Acc: 0.9826
Epoch [16/150] | Loss: 0.2070 | Acc: 0.9790
Epoch [17/150] | Loss: 0.0519 | Acc: 0.9888
Epoch [18/150] | Loss: 0.0386 | Acc: 0.9938
Epoch [19/150] | Loss: 0.0236 | Acc: 0.9951
Epoch [20/150] | Loss: 0.0382 | Acc: 0.9902
Epoch [21/150] | Loss: 0.0204 | Acc: 0.9955
Epoch [22/150] | Loss: 0.1393 | Acc: 0.

## Evaluation
---

In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.stats as stats

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    accuracy_score
)

from torch.utils.data import DataLoader


def k_of_n_probability(p, k=3, n=5):
    """
    Computes binomial probability of >= k anomaly detections in n consecutive frame predictions.
    Formula: sum_{i=k}^n binom(n, i) * p^i * (1 - p)^(n - i)
    For k=3, n=5:
      P(>=3 of 5) = 10*p^3*(1-p)^2 + 5*p^4*(1-p) + p^5
    """
    return float(1.0 - stats.binom.cdf(k - 1, n, p))


def evaluate_model(model,
                   test_normal_dir,
                   anomaly_dir,
                   id_mapping,
                   device,
                   scoring_method="negative_logit",
                   target_fnr=0.066,
                   window_frames=5,
                   window_thresh=3,
                   batch_size=16):
    """
    Evaluates SW-WaveNet model enforcing Frame-Level FNR <= target_fnr dynamically per machine.
    Derives the 2 key Processing Layer factors via Binomial Consensus (>=3 of 5 frames):
      1. Missed Alarm Rate = 1 - P(>=3 of 5 | Anomaly)
      2. False Alarm Rate  = P(>=3 of 5 | Normal)
    """
    model.eval()

    normal_set = SafeMachineDataset(test_normal_dir, id_mapping)
    anomaly_set = SafeMachineDataset(anomaly_dir, id_mapping)

    y_true = []
    y_scores = []
    machine_ids = []

    index_to_id = {v: k for k, v in id_mapping.items()}
    target_fnr_pct = target_fnr * 100.0
    target_tpr_pct = (1.0 - target_fnr) * 100.0

    print(f"Loaded {len(normal_set)} normal test samples and {len(anomaly_set)} anomaly samples.")
    print(f"Target Frame-Level FNR: <={target_fnr_pct:.2f}% (Min Detection >= {target_tpr_pct:.2f}%)")
    print(f"Processing Layer Rule : ALARM is set if >={window_thresh} of {window_frames} predictions are ANOMALY")
    print(f"Device: {device} | Scoring: {scoring_method}")

    normal_loader = DataLoader(normal_set, batch_size=batch_size, shuffle=False)
    anomaly_loader = DataLoader(anomaly_set, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for audio, labels in normal_loader:
            audio = audio.to(device)
            if isinstance(model, torch.jit.ScriptModule):
                logits = model(audio)
            else:
                logits = model(audio, None)
            for b in range(audio.size(0)):
                l_idx = labels[b].item()
                t_logit = logits[b, l_idx].item()
                y_true.append(0)
                y_scores.append(-t_logit if scoring_method == "negative_logit" else 1.0 - (t_logit/30.0))
                machine_ids.append(index_to_id[l_idx])

        for audio, labels in anomaly_loader:
            audio = audio.to(device)
            if isinstance(model, torch.jit.ScriptModule):
                logits = model(audio)
            else:
                logits = model(audio, None)
            for b in range(audio.size(0)):
                l_idx = labels[b].item()
                t_logit = logits[b, l_idx].item()
                y_true.append(1)
                y_scores.append(-t_logit if scoring_method == "negative_logit" else 1.0 - (t_logit/30.0))
                machine_ids.append(index_to_id[l_idx])

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    machine_ids = np.array(machine_ids)
    unique_mids = np.unique(machine_ids)

    per_machine_results = {}
    frame_table_rows = []
    layer_table_rows = []

    for mid in unique_mids:
        mask = (machine_ids == mid)
        m_true = y_true[mask]
        m_scores = y_scores[mask]

        m_auc = roc_auc_score(m_true, m_scores)
        m_pauc = roc_auc_score(m_true, m_scores, max_fpr=0.1)
        m_fpr, m_tpr, m_thresholds = roc_curve(m_true, m_scores)
        m_fnr = 1.0 - m_tpr

        # Find machine threshold satisfying Frame FNR <= target_fnr
        valid_fnr_idx = np.where(m_fnr <= target_fnr)[0]
        fnr_idx = valid_fnr_idx[0] if len(valid_fnr_idx) > 0 else 0
        m_th = m_thresholds[fnr_idx]

        # 1. Single-Frame Metrics
        m_preds = (m_scores >= m_th).astype(int)
        m_cm = confusion_matrix(m_true, m_preds)
        tn, fp, fn_count, tp = m_cm.ravel()
        frame_tpr = tp / (tp + fn_count) if (tp + fn_count) > 0 else 0       # Frame True Detection Rate
        frame_fnr = fn_count / (tp + fn_count) if (tp + fn_count) > 0 else 0 # Frame Miss Rate (FNR)
        frame_fpr = fp / (tn + fp) if (tn + fp) > 0 else 0                   # Frame False Alarm Rate (FPR)
        frame_acc = accuracy_score(m_true, m_preds)

        # 2. Processing Layer Metrics (Binomial Formula for >=3 of 5 frames)
        layer_true_alarm = k_of_n_probability(frame_tpr, k=window_thresh, n=window_frames) # True Alarm Detection Rate
        layer_miss_rate  = 1.0 - layer_true_alarm                                          # Factor 1: Missed Alarm Rate
        layer_false_alarm = k_of_n_probability(frame_fpr, k=window_thresh, n=window_frames) # Factor 2: False Alarm Rate

        per_machine_results[mid] = {
            "threshold": m_th,
            "target_logit": -m_th if scoring_method == "negative_logit" else (1.0 - m_th) * 30.0,
            "frame_tpr": frame_tpr,
            "frame_fnr": frame_fnr,
            "frame_fpr": frame_fpr,
            "frame_acc": frame_acc,
            "layer_true_alarm": layer_true_alarm,
            "layer_miss_rate": layer_miss_rate,
            "layer_false_alarm": layer_false_alarm,
            "tp": tp,
            "fn": fn_count,
            "fp": fp,
            "tn": tn,
            "total_anomaly": tp + fn_count,
            "total_normal": tn + fp,
            "auc": m_auc,
            "pauc": m_pauc,
            "fpr_curve": m_fpr,
            "tpr_curve": m_tpr,
            "thresh_curve": m_thresholds,
            "cm": m_cm
        }

        # Frame-level row
        frame_table_rows.append({
            "Machine": f"ID {mid:02d}",
            "Threshold (tau)": f"{m_th:.4f}",
            "Target Logit": f"{-m_th:.2f}",
            "Frame TPR (Detection)": f"{frame_tpr*100:.2f}% ({tp}/{tp+fn_count})",
            "Frame FNR (Miss Rate)": f"{frame_fnr*100:.2f}% ({fn_count}/{tp+fn_count})",
            "Frame FPR (False Alarm)": f"{frame_fpr*100:.2f}% ({fp}/{tn+fp})",
            "Accuracy": f"{frame_acc*100:.2f}%",
            f"FNR <= {target_fnr_pct:.1f}%": "PASSED" if frame_fnr <= target_fnr else "FAILED"
        })

        # Processing Layer row (Derived 2 Factors)
        layer_table_rows.append({
            "Machine": f"ID {mid:02d}",
            "Frame Detection": f"{frame_tpr*100:.2f}%",
            "Frame Miss (FNR)": f"{frame_fnr*100:.2f}%",
            "Frame False Alarm": f"{frame_fpr*100:.2f}%",
            f"Layer True Alarm (>={window_thresh}/{window_frames})": f"{layer_true_alarm*100:.3f}%",
            f"MISSED ALARM RATE (1 - True Alarm)": f"{layer_miss_rate*100:.3f}%",
            f"FALSE ALARM RATE (P(Alarm|Normal))": f"{layer_false_alarm*100:.3f}%"
        })

    # ==========================================================
    # PRINT FORMULA & SUMMARY TABLES
    # ==========================================================
    print("\n" + "="*115)
    print(f"  1. FRAME-LEVEL PERFORMANCE (ENFORCING FRAME FNR <= {target_fnr_pct:.2f}% PER MACHINE)")
    print("="*115)
    df_frame = pd.DataFrame(frame_table_rows)
    print(df_frame.to_string(index=False))

    print("\n" + "="*115)
    print(f"  2. PROCESSING LAYER CONSENSUS (>={window_thresh} OF {window_frames} PREDICTIONS = ALARM)")
    print("="*115)
    print("  BINOMIAL CONSENSUS FORMULA:")
    print(f"    * P(Alarm) = P(Anomalies >= {window_thresh} of {window_frames}) = 10*p^3*(1-p)^2 + 5*p^4*(1-p) + p^5")
    print("    * Factor 1: MISSED ALARM RATE = 1 - P(Alarm | p = Frame TPR)")
    print("    * Factor 2: FALSE ALARM RATE  = P(Alarm | p = Frame FPR)")
    print("-"*115)
    df_layer = pd.DataFrame(layer_table_rows)
    print(df_layer.to_string(index=False))
    print("="*115)

    # ==========================================================
    # DYNAMIC VISUALIZATIONS (3-PANEL PER MACHINE)
    # ==========================================================
    n_mids = len(unique_mids)
    fig, axes = plt.subplots(n_mids, 3, figsize=(18, 4.5 * n_mids))
    if n_mids == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, mid in enumerate(unique_mids):
        res = per_machine_results[mid]
        m_mask = (machine_ids == mid)
        m_scores = y_scores[m_mask]
        m_true = y_true[m_mask]

        # Col 1: Error Rates (FNR & FPR) vs Threshold
        thresh_s = np.sort(res["thresh_curve"])
        f_curve, p_curve = [], []
        for t in thresh_s:
            p = (m_scores >= t).astype(int)
            cm_t = confusion_matrix(m_true, p)
            tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
            f_curve.append(fn_t / (tp_t + fn_t))
            p_curve.append(fp_t / (tn_t + fp_t))

        axes[i, 0].plot(thresh_s, np.array(f_curve)*100, color="crimson", lw=2.2, label="Frame Miss Rate (FNR %)")
        axes[i, 0].plot(thresh_s, np.array(p_curve)*100, color="dodgerblue", lw=2.2, label="Frame False Alarm (FPR %)")
        axes[i, 0].axhline(target_fnr_pct, color="darkred", linestyle=":", lw=1.8, label=f"Target Max FNR ({target_fnr_pct:.1f}%)")
        axes[i, 0].axvline(res["threshold"], color="black", linestyle="--", lw=2, label=f"$\tau_{mid}$ = {res['threshold']:.2f}")
        axes[i, 0].set_xlabel("Anomaly Score (-Logit)")
        axes[i, 0].set_ylabel("Error Rate (%)")
        axes[i, 0].set_title(f"Machine ID {mid:02d}: Error Rates vs. Threshold (Target FNR <= {target_fnr_pct:.1f}%)")
        axes[i, 0].legend(loc="center right", fontsize=9)
        axes[i, 0].grid(True, alpha=0.3)

        # Col 2: ROC Curve & Operating Point
        axes[i, 1].plot(res["fpr_curve"]*100, res["tpr_curve"]*100, color="darkorange", lw=2.2, label=f"ROC (AUC = {res['auc']*100:.2f}%)")
        axes[i, 1].plot([0, 100], [0, 100], color="navy", lw=1.5, linestyle="--")
        axes[i, 1].axhline(target_tpr_pct, color="darkred", linestyle=":", lw=1.5, label=f"Min TPR ({target_tpr_pct:.1f}%) for FNR<={target_fnr_pct:.1f}%")
        axes[i, 1].plot(res["frame_fpr"]*100, res["frame_tpr"]*100, marker='*', markersize=14, color="red",
                        label=f"Operating Point (FNR={res['frame_fnr']*100:.1f}%, FPR={res['frame_fpr']*100:.1f}%)")
        axes[i, 1].set_xlabel("False Positive Rate (FPR %)")
        axes[i, 1].set_ylabel("True Positive Rate (TPR %)")
        axes[i, 1].set_title(f"Machine ID {mid:02d}: ROC Operating Point (TPR = {res['frame_tpr']*100:.1f}%)")
        axes[i, 1].legend(loc="lower right", fontsize=9)
        axes[i, 1].grid(True, alpha=0.3)

        # Col 3: Comparison Bar Chart (Frame-Level vs Processing Layer 3 of 5)
        metrics = ["True Alarm Rate", "Missed Alarm Rate", "False Alarm Rate"]
        frame_bars = [res["frame_tpr"]*100, res["frame_fnr"]*100, res["frame_fpr"]*100]
        layer_bars = [res["layer_true_alarm"]*100, res["layer_miss_rate"]*100, res["layer_false_alarm"]*100]

        x = np.arange(len(metrics))
        width = 0.35
        axes[i, 2].bar(x - width/2, frame_bars, width, label="Frame-Level (1 Frame)", color="steelblue")
        axes[i, 2].bar(x + width/2, layer_bars, width, label=f"Processing Layer (>={window_thresh}/{window_frames})", color="coral")
        axes[i, 2].set_xticks(x)
        axes[i, 2].set_xticklabels(metrics)
        axes[i, 2].set_ylabel("Rate (%)")
        axes[i, 2].set_title(f"Machine ID {mid:02d}: Frame vs. Layer Rates (>={window_thresh}/{window_frames})")
        axes[i, 2].legend(fontsize=9)
        axes[i, 2].grid(True, alpha=0.3)

        for c_idx, val in enumerate(frame_bars):
            axes[i, 2].text(c_idx - width/2, val + 1.2, f"{val:.1f}%", ha='center', fontsize=8, fontweight='bold')
        for c_idx, val in enumerate(layer_bars):
            axes[i, 2].text(c_idx + width/2, val + 1.2, f"{val:.1f}%", ha='center', fontsize=8, fontweight='bold')

    plt.tight_layout()
    plt.show()

    return {
        "per_machine": per_machine_results,
        "frame_summary": df_frame,
        "layer_summary": df_layer
    }


### Execute Evaluation
---

In [ ]:
# ============================================================
# DYNAMIC EVALUATION: CHANGE target_fnr AS DESIRED
# ============================================================
# You can adjust target_fnr (e.g. 0.066 for 6.6%, 0.05 for 5.0%, 0.03 for 3.0%)
TARGET_MAX_FNR = 0.066  # Maximum acceptable Frame-Level FNR (6.6%)

eval_results = evaluate_model(
    model=model,
    test_normal_dir=TEST_NORMAL_DIR,
    anomaly_dir=ANOMALY_DIR,
    id_mapping=id_mapping,
    device=device,
    scoring_method="negative_logit",
    target_fnr=TARGET_MAX_FNR,   # Adjust here to test different sensitivity levels
    window_frames=5,            # Sliding buffer of 5 predictions (25 sec)
    window_thresh=3,            # >= 3 of 5 predictions classified as anomaly triggers ALARM
    batch_size=16
)


## Export Model (to cpu)
---

In [7]:
import torch
import torch.nn as nn

class InferenceWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, raw_audio):
        # call original model without labels
        return self.model(raw_audio, None)

In [8]:
def export_to_cpu(model, save_path="sw_wavenet_traced_cpu.pt"):

    model.eval()
    model.cpu()

    # Wrap model for inference
    inference_model = InferenceWrapper(model)
    inference_model.eval()
    inference_model.cpu()

    example = torch.randn(1, 1, 160000)

    # ✅ Now trace only tensor input
    traced = torch.jit.trace(inference_model, example)

    traced.save(save_path)

    print("✅ TorchScript CPU model saved at:", save_path)

In [9]:
export_to_cpu(model)

✅ TorchScript CPU model saved at: sw_wavenet_traced_cpu.pt
